In [31]:
import pandas as pd
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
import string

In [32]:
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [33]:
data = pd.read_csv('../data/raw/spam.csv', encoding='latin-1')
data = data.rename(columns={'v1': 'label', 'v2': 'text'})
data = data.drop(columns=['Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'])
print(data)

     label                                               text
0      ham  Go until jurong point, crazy.. Available only ...
1      ham                      Ok lar... Joking wif u oni...
2     spam  Free entry in 2 a wkly comp to win FA Cup fina...
3      ham  U dun say so early hor... U c already then say...
4      ham  Nah I don't think he goes to usf, he lives aro...
...    ...                                                ...
5567  spam  This is the 2nd time we have tried 2 contact u...
5568   ham              Will Ì_ b going to esplanade fr home?
5569   ham  Pity, * was in mood for that. So...any other s...
5570   ham  The guy did some bitching but I acted like i'd...
5571   ham                         Rofl. Its true to its name

[5572 rows x 2 columns]


In [34]:
# Preprocessing into lower case
data['text'] = data['text'].str.lower()
print(data.head())

  label                                               text
0   ham  go until jurong point, crazy.. available only ...
1   ham                      ok lar... joking wif u oni...
2  spam  free entry in 2 a wkly comp to win fa cup fina...
3   ham  u dun say so early hor... u c already then say...
4   ham  nah i don't think he goes to usf, he lives aro...


In [35]:
# Removing punctuation
def remove_punctuation(text):
    return text.translate(str.maketrans('', '', string.punctuation))

data['text'] = data['text'].apply(remove_punctuation)
print(data.head())

  label                                               text
0   ham  go until jurong point crazy available only in ...
1   ham                            ok lar joking wif u oni
2  spam  free entry in 2 a wkly comp to win fa cup fina...
3   ham        u dun say so early hor u c already then say
4   ham  nah i dont think he goes to usf he lives aroun...


In [36]:
# Removing alphanumeric characters
data['text'] = data['text'].apply(lambda x: re.sub(r'a-zA-Z0-9', '', x))
print(data.head())

  label                                               text
0   ham  go until jurong point crazy available only in ...
1   ham                            ok lar joking wif u oni
2  spam  free entry in 2 a wkly comp to win fa cup fina...
3   ham        u dun say so early hor u c already then say
4   ham  nah i dont think he goes to usf he lives aroun...


In [37]:
# Tokenization
data['tokens'] = data['text'].apply(word_tokenize)
print(data.head())

  label                                               text  \
0   ham  go until jurong point crazy available only in ...   
1   ham                            ok lar joking wif u oni   
2  spam  free entry in 2 a wkly comp to win fa cup fina...   
3   ham        u dun say so early hor u c already then say   
4   ham  nah i dont think he goes to usf he lives aroun...   

                                              tokens  
0  [go, until, jurong, point, crazy, available, o...  
1                     [ok, lar, joking, wif, u, oni]  
2  [free, entry, in, 2, a, wkly, comp, to, win, f...  
3  [u, dun, say, so, early, hor, u, c, already, t...  
4  [nah, i, dont, think, he, goes, to, usf, he, l...  


In [38]:
stop_words = set(stopwords.words('english'))

data['tokens'] = data['tokens'].apply(
    lambda words: [word for word in words if word not in stop_words]
)
print(data.head())

  label                                               text  \
0   ham  go until jurong point crazy available only in ...   
1   ham                            ok lar joking wif u oni   
2  spam  free entry in 2 a wkly comp to win fa cup fina...   
3   ham        u dun say so early hor u c already then say   
4   ham  nah i dont think he goes to usf he lives aroun...   

                                              tokens  
0  [go, jurong, point, crazy, available, bugis, n...  
1                     [ok, lar, joking, wif, u, oni]  
2  [free, entry, 2, wkly, comp, win, fa, cup, fin...  
3      [u, dun, say, early, hor, u, c, already, say]  
4  [nah, dont, think, goes, usf, lives, around, t...  


In [39]:
lemmatizer = WordNetLemmatizer()

data['tokens'] = data['tokens'].apply(
    lambda words: [lemmatizer.lemmatize(word) for word in words]
)
print(data.head())

  label                                               text  \
0   ham  go until jurong point crazy available only in ...   
1   ham                            ok lar joking wif u oni   
2  spam  free entry in 2 a wkly comp to win fa cup fina...   
3   ham        u dun say so early hor u c already then say   
4   ham  nah i dont think he goes to usf he lives aroun...   

                                              tokens  
0  [go, jurong, point, crazy, available, bugis, n...  
1                     [ok, lar, joking, wif, u, oni]  
2  [free, entry, 2, wkly, comp, win, fa, cup, fin...  
3      [u, dun, say, early, hor, u, c, already, say]  
4  [nah, dont, think, go, usf, life, around, though]  


In [40]:
data['clean_text'] = data['tokens'].apply(lambda x: " ".join(x))
print(data.head())

  label                                               text  \
0   ham  go until jurong point crazy available only in ...   
1   ham                            ok lar joking wif u oni   
2  spam  free entry in 2 a wkly comp to win fa cup fina...   
3   ham        u dun say so early hor u c already then say   
4   ham  nah i dont think he goes to usf he lives aroun...   

                                              tokens  \
0  [go, jurong, point, crazy, available, bugis, n...   
1                     [ok, lar, joking, wif, u, oni]   
2  [free, entry, 2, wkly, comp, win, fa, cup, fin...   
3      [u, dun, say, early, hor, u, c, already, say]   
4  [nah, dont, think, go, usf, life, around, though]   

                                          clean_text  
0  go jurong point crazy available bugis n great ...  
1                            ok lar joking wif u oni  
2  free entry 2 wkly comp win fa cup final tkts 2...  
3                u dun say early hor u c already say  
4           nah

In [41]:
processed_data = data.to_csv('../data/processed/processed_spam.csv', index=False)